# 从字节对到可还原的分词器

将 UTF-8 文本转成字节 ID，统计相邻对，再学习替换规则。后续单元逐步引入预分词边界、特殊标记以及保存和加载。

In [1]:
def count_pairs(ids):
    pair_counts = {}
    for i in range(len(ids) - 1):
        pair = (ids[i], ids[i + 1])
        pair_counts[pair] = pair_counts.get(pair,0) + 1

    return pair_counts

In [2]:
print(count_pairs([97, 98, 97, 98, 97, 98]))
print(count_pairs([97, 97, 97]))

{(97, 98): 3, (98, 97): 2}
{(97, 97): 2}


In [3]:
def merge(ids, pair, new_id):
    new_ids = []
    i = 0
    while i < len(ids):
        if i + 1 < len(ids) and (ids[i], ids[i + 1]) == pair:
            new_ids.append(new_id)
            i = i + 2
        else:
            new_ids.append(ids[i])
            i = i + 1
    return new_ids

In [4]:
print(merge([97, 98, 97, 98, 97, 98], (97, 98), 256))
print(merge([97, 97, 97], (97, 97), 256))
print(merge([98, 97], (97, 98), 256))

[256, 256, 256]
[256, 97]
[98, 97]


In [5]:
ids = [97, 98, 97, 98, 97, 98]

pair_counts = count_pairs(ids)
best_pair = max(pair_counts, key=pair_counts.get)

print("选中的组合：", best_pair)

ids = merge(ids,best_pair,256)
print(ids)

选中的组合： (97, 98)
[256, 256, 256]


In [6]:
# 重新从原文开始，避免接着之前已经合并过的 ids 操作
text = "ababab"
ids = list(text.encode("utf-8"))

# 初始化基础词表
vocab = {}
for token_id in range(256):
    vocab[token_id] = bytes([token_id])

merges = {}
num_merges = 2

for step in range(num_merges):
    # 1. 统计当前序列的相邻组合
    pair_counts = count_pairs(ids)

    # 不足两个 token 时，没有可以合并的组合
    if not pair_counts:
        break

    # 2. 选择最高频组合
    best_pair = max(pair_counts, key=pair_counts.get)

    # 3. 每轮使用一个新的编号：256、257、258……
    new_id = 256 + step

    # 4. 记录新 token 的内容，以及产生它的规则
    vocab[new_id] = vocab[best_pair[0]] + vocab[best_pair[1]]
    merges[best_pair] = new_id

    # 5. 更新序列，供下一轮重新统计
    ids = merge(ids, best_pair, new_id)

    print(f"第 {step + 1} 轮：{best_pair} → {new_id}")
    print("新 token 的内容：", vocab[new_id])
    print("合并后：", ids)

第 1 轮：(97, 98) → 256
新 token 的内容： b'ab'
合并后： [256, 256, 256]
第 2 轮：(256, 256) → 257
新 token 的内容： b'abab'
合并后： [257, 256]


In [7]:
text = "ababab"
ids = list(text.encode("utf-8"))
vocab = {}
for token_id in range(256):
    vocab[token_id] = bytes([token_id])

merges = {}
num_merges = 2

for step in range(num_merges):
    pair_counts = count_pairs(ids)

    if not pair_counts:
        break

    best_pair = max(pair_counts,key=pair_counts.get)

    new_id = 256 + step

    vocab[new_id] = vocab[best_pair[0]] + vocab[best_pair[1]]
    merges[best_pair] = new_id

    ids = merge(ids, best_pair, new_id)

    print(f"第 {step + 1} 轮：{best_pair} → {new_id}")
    print("新 token 的内容：", vocab[new_id])
    print("合并后：", ids)

第 1 轮：(97, 98) → 256
新 token 的内容： b'ab'
合并后： [256, 256, 256]
第 2 轮：(256, 256) → 257
新 token 的内容： b'abab'
合并后： [257, 256]


In [8]:
def encode(text,merges):
    ids = list(text.encode("utf-8"))

    for pair, new_id in merges.items():
        ids = merge(ids, pair, new_id)

    return ids

In [9]:
print(encode("ababac", merges))

[257, 97, 99]


In [10]:
def decode(ids,vocab):
    origin = []

    for token_id in ids:
        piece = vocab[token_id]
        origin.append(piece)

    data = b"".join(origin)
    return data.decode("utf-8")

In [11]:
text = "ababac，你好！"

token_ids = encode(text, merges)
restored = decode(token_ids, vocab)

print(restored)
print(restored == text)

ababac，你好！
True


## 按预分词边界训练

先划分文本片段，再逐段统计和合并。用带连续空格、换行和中文的句子检查能否还原原文。

In [12]:
import re

text = "Hi,  AI!\n你好。"

pattern = r"\w+|\s+|[^\w\s]"
parts = re.findall(pattern, text)

print(parts)
print("".join(parts) == text)

['Hi', ',', '  ', 'AI', '!', '\n', '你好', '。']
True


In [13]:
import re


def pretokenize(text):
    return re.findall(r"\w+|\s+|[^\w\s]", text)

In [14]:
def text_to_chunks(text):
    pieces = pretokenize(text)

    chunks = []
    for piece in pieces:
        chunks.append(list(piece.encode("utf-8")))

    return chunks

In [15]:
def count_chunk_pairs(chunks):
    total_counts = {}
    for chunk in chunks:
        count = count_pairs(chunk)

        for chunk_pair, chunk_count in count.items():
            total_counts[chunk_pair] = total_counts.get(chunk_pair,0) + chunk_count

    return total_counts

In [16]:
def merge_chunks(chunks, pair, new_id):
    new_chunks = []
    for chunk in chunks:
        new_chunk = merge(chunk, pair, new_id)
        new_chunks.append(new_chunk)
    chunks = new_chunks

    return new_chunks

In [17]:
vocab = {i: bytes([i]) for i in range(256)}
merges = {}

pieces = pretokenize("ab ab!")

chunks = []
for piece in pieces:
    chunks.append(list(piece.encode("utf-8")))

for step in range(num_merges):
    total_counts = {}# 本轮重新统计
    for chunk in chunks:
        count = count_pairs(chunk)

        for chunk_pair, chunk_count in count.items():
            total_counts[chunk_pair] = (
                total_counts.get(chunk_pair, 0) + chunk_count
        )
    # 判断是否结束、选组合、记录规则
    # 不足两个 token 时，没有可以合并的组合
    if not total_counts:
        break

    # 2. 选择最高频组合
    best_pair = max(total_counts, key=total_counts.get)

    # 3. 每轮使用一个新的编号：256、257、258……
    new_id = 256 + step

    # 4. 记录新 token 的内容，以及产生它的规则
    vocab[new_id] = vocab[best_pair[0]] + vocab[best_pair[1]]
    merges[best_pair] = new_id

    # 5. 更新序列，供下一轮重新统计

    new_chunks = []     # 本轮重新收集合并结果
    for chunk in chunks:
        new_chunk = merge(chunk, best_pair, new_id)
        new_chunks.append(new_chunk)
    chunks = new_chunks

In [18]:
def encode(text, merges):
    result = []
    chunks = text_to_chunks(text)
    for chunk_pair, chunk_id in merges.items():
        chunks = merge_chunks(chunks, chunk_pair, chunk_id)
    for chunk in chunks:
        result.extend(chunk)
    return result

In [19]:
print(encode("ab ab!", merges))

[256, 32, 256, 33]


In [20]:
text = "ab  ab!\n你好。"

token_ids = encode(text, merges)
restored = decode(token_ids, vocab)

print(token_ids)
print(repr(restored))
print(restored == text)

[256, 32, 32, 256, 33, 10, 228, 189, 160, 229, 165, 189, 227, 128, 130]
'ab  ab!\n你好。'
True


## 单个特殊标记

把结束标记当作一个 ID，在普通文本中继续应用 BPE 合并规则。

In [21]:
text = "ab<|endoftext|>ab"
special = "<|endoftext|>"

pattern = "(" + re.escape(special) + ")"
parts = re.split(pattern, text)

print(parts)

['ab', '<|endoftext|>', 'ab']


In [22]:
assert "".join(parts) == text

In [23]:
special = "<|endoftext|>"
special_id = max(vocab) + 1

vocab[special_id] = special.encode("utf-8")

In [24]:
def split_special(text, special_tokens):
    if not special_tokens:
        return [text]

    ordered_special = sorted(special_tokens, key=len, reverse=True)

    escaped_specials = []
    for special in ordered_special:
        escaped_specials.append(re.escape(special))  # 对当前标记调用 re.escape

    body = "|".join(escaped_specials)
    pattern = "(" + body +")"  # 在 body 外面加一对捕获括号

    parts = re.split(pattern, text)

    return parts

In [25]:
def encode_with_special(text, merges, special_ids):
    result = []

    parts = split_special(text, list(special_ids))


    for part in parts:
        if part in special_ids:
            result.append(special_ids[part])
        else:
            result.extend(encode(part,merges))

    return result

In [26]:
text = "ab<|endoftext|>ab"
token_ids = encode_with_special(text, merges, {special: special_id})
print(token_ids)

[256, 257, 256]


In [27]:
print(decode(token_ids, vocab))
print(decode(token_ids, vocab) == text)

ab<|endoftext|>ab
True


In [28]:
chunks = []

for part in parts:
    if part != special:
        chunks.extend(text_to_chunks(part))

In [29]:
text = "ababab<|endoftext|>ab"

#① 在训练循环之前，重新建立初始词表，并先登记特殊标记。

vocab_size = 260
special = "<|endoftext|>"

vocab = {}
for token_id in range(256):
    vocab[token_id] = bytes([token_id])

merges = {}

special_id = len(vocab)
vocab[special_id] = special.encode("utf-8")

pattern = "(" + re.escape(special) + ")"
parts = re.split(pattern, text)

chunks = []

for part in parts:
    if part != special:
        chunks.extend(text_to_chunks(part))

num_merges = vocab_size - len(vocab)

for step in range(num_merges):
    pair_counts = count_chunk_pairs(chunks)

    if not pair_counts:
        break

    best_pair = max(pair_counts,key=pair_counts.get)
    new_id = len(vocab)

    vocab[new_id] = vocab[best_pair[0]] + vocab[best_pair[1]]
    merges[best_pair] = new_id

    chunks = merge_chunks(chunks, best_pair, new_id)
    print(new_id, vocab[new_id], len(vocab))

print("最终词表大小：", len(vocab))
print("合并规则：", merges)

257 b'ab' 258
258 b'abab' 259
259 b'ababab' 260
最终词表大小： 260
合并规则： {(97, 98): 257, (257, 257): 258, (258, 257): 259}


In [30]:
best_pair = max(
    pair_counts,
    key=lambda pair: (
        pair_counts[pair],
        vocab[pair[0]],
        vocab[pair[1]],
    ),
)

In [31]:
merge_rules = []

for pair in merges:
    # 查出 pair 左右两个 ID 对应的 bytes，组成一个元组
    byte_pair = (vocab[pair[0]], vocab[pair[1]])
    merge_rules.append(byte_pair)

print(merge_rules)

[(b'a', b'b'), (b'ab', b'ab'), (b'abab', b'ab')]


## 训练接口与多个特殊标记

当特殊标记彼此重叠时，先识别较长的标记。

In [32]:
def train_bpe(text, vocab_size, special_tokens):
    # 1. 创建基础 vocab 和空 merges
    vocab = {}
    for token_id in range(256):
        vocab[token_id] = bytes([token_id])

    merges = {}

    # 2. 登记特殊标记，得到 special_ids
    special_ids = {}

    for special in special_tokens:
        special_id = len(vocab)
        vocab[special_id] = special.encode("utf-8")
        special_ids[special] = special_id

    # 3. 切开特殊标记，把普通文本转换成 chunks
    parts = split_special(text, special_tokens)

    chunks = []

    for part in parts:
        if part not in special_ids:
            chunks.extend(text_to_chunks(part))
    # 4. 计算剩余合并次数，执行训练循环
    num_merges = vocab_size - len(vocab)
    for step in range(num_merges):
        pair_counts = count_chunk_pairs(chunks)

        if not pair_counts:
            break

        best_pair = max(
        pair_counts,
        key=lambda pair: (
            pair_counts[pair],
            vocab[pair[0]],
            vocab[pair[1]]
        )
        )

        new_id = len(vocab)

        vocab[new_id] = vocab[best_pair[0]] + vocab[best_pair[1]]
        merges[best_pair] = new_id

        chunks = merge_chunks(chunks, best_pair, new_id)
    # 5. 返回训练结果
    return vocab, merges, special_ids

In [33]:
text = "ababab<|endoftext|>ab"
special_tokens = ["<|endoftext|>"]

vocab, merges, special_ids = train_bpe(
    text, 260, special_tokens
)

print(special_ids, len(vocab))
print(merges)

token_ids = encode_with_special(text, merges, special_ids)
print(token_ids)
print(decode(token_ids, vocab) == text)

{'<|endoftext|>': 256} 260
{(97, 98): 257, (257, 257): 258, (258, 257): 259}
[259, 256, 257]
True


In [34]:
demo_vocab = {}
for token_id in range(256):
    demo_vocab[token_id] = bytes([token_id])

special_tokens = ["<|endoftext|>", "<|pad|>"]
special_ids = {}

for special in special_tokens:
    special_id = len(demo_vocab)
    demo_vocab[special_id] = special.encode("utf-8")
    special_ids[special] = special_id

print(special_ids)
print(len(demo_vocab))

{'<|endoftext|>': 256, '<|pad|>': 257}
258


In [35]:
text = "x<A><B>y"
special_tokens = ["<A>", "<A><B>"]

ordered_special = sorted(special_tokens, key=len, reverse=True)

escaped_specials = []
for special in ordered_special:
    escaped_specials.append(re.escape(special))  # 对当前标记调用 re.escape

body = "|".join(escaped_specials)
pattern = "(" + body +")"  # 在 body 外面加一对捕获括号

parts = re.split(pattern, text)
print(parts)

['x', '<A><B>', 'y']


In [36]:
print(split_special("x<A><B>y", ["<A>", "<A><B>"]))

['x', '<A><B>', 'y']


In [37]:
special_tokens = ["<|endoftext|>", "<|pad|>"]
text = "ababab<|endoftext|>ab<|pad|>"

vocab, merges, special_ids = train_bpe(text, 260, special_tokens)

print(special_ids)
print(len(vocab))
print(merges)

{'<|endoftext|>': 256, '<|pad|>': 257}
260
{(97, 98): 258, (258, 258): 259}


In [38]:
token_ids = encode_with_special(text, merges, special_ids)

print(token_ids)
print(decode(token_ids, vocab) == text)

[259, 258, 256, 258, 257]
True


## Unicode 预分词和持久化

从字符类与英文缩写构造预分词表达式，把词表和合并规则写入文件，再从该文件读回。

In [39]:
import regex

pattern = r" ?\p{L}+| ?\p{N}+"
parts = regex.findall(pattern, "abc123 abc 456")
print(parts)

['abc', '123', ' abc', ' 456']


In [40]:
pattern = r" ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+"

parts = regex.findall(pattern, "Hi, 2026!!! 😊")
print(parts)

['Hi', ',', ' 2026', '!!!', ' 😊']


In [41]:
contraction_pattern = r"'(?:[sdmt]|ll|ve|re)"
pattern_with_contractions = contraction_pattern + "|" + pattern

print(regex.findall(pattern_with_contractions, "I'm we're"))

['I', "'m", ' we', "'re"]


In [42]:
full_pattern = pattern_with_contractions + r"|\s+(?!\S)|\s+"

text = "hello  hello"
parts = regex.findall(full_pattern, text)

print(parts)
print("".join(parts) == text)


def pretokenize(text):
    return regex.findall(full_pattern, text)

['hello', ' ', ' hello']
True


In [43]:
special_tokens = ["<|endoftext|>", "<|pad|>"]
train_text = "ab ab<|endoftext|> ab!<|pad|>"

vocab, merges, special_ids = train_bpe(
    train_text, 260, special_tokens
)

token_ids = encode_with_special(train_text, merges, special_ids)

print(merges)
print(vocab[259])
print(token_ids)
print(decode(token_ids, vocab) == train_text)

{(97, 98): 258, (32, 258): 259}
b' ab'
[258, 259, 256, 259, 33, 257]
True


In [44]:
import pickle
from pathlib import Path
from tempfile import TemporaryDirectory

state_dir = TemporaryDirectory()
state_path = Path(state_dir.name) / "tokenizer.pkl"

state = {
    "vocab": vocab,
    "merges": merges,
    "special_ids": special_ids,
    "pattern": full_pattern
}

with open(state_path,"wb") as f:
    pickle.dump(state, f)

In [45]:
with open(state_path, "rb") as f:
    loaded = pickle.load(f)

loaded_vocab =loaded["vocab"]
loaded_merges = loaded["merges"]
loaded_special_ids = loaded["special_ids"]
full_pattern = loaded["pattern"]

loaded_ids = encode_with_special(
    train_text, loaded_merges, loaded_special_ids
)

print(loaded_ids == token_ids)
print(decode(loaded_ids, loaded_vocab) == train_text)
print("保存前：", token_ids)
print("加载后：", loaded_ids)
state_dir.cleanup()

True
True
保存前： [258, 259, 256, 259, 33, 257]
加载后： [258, 259, 256, 259, 33, 257]


## 分词器对象

把配置、编码、解码和逐条编码的操作放进同一对象。

In [46]:
class Tokenizer:
    def __init__(self, vocab, merges, special_ids, pattern):
        self.vocab = vocab
        self.merges = merges
        self.special_ids = special_ids
        self.pattern = pattern

    def decode(self, ids):
        return decode(ids, self.vocab)

In [47]:
class Tokenizer:
    def __init__(self, vocab, merges, special_ids, pattern):
        self.vocab = vocab
        self.merges = merges
        self.special_ids = special_ids
        self.pattern = pattern

    def encode(self, text):
        result = []

        # 1. 先识别特殊标记
        parts = split_special(text, list(self.special_ids))

        for part in parts:
            if part in self.special_ids:
                # 特殊标记直接查编号
                result.append(self.special_ids[part])
            else:
                # 2. 对普通文本进行预分词
                pieces = regex.findall(self.pattern, part)

                for piece in pieces:
                    # 3. 每个块转换成初始字节 ID
                    ids = list(piece.encode("utf-8"))

                    # 4. 按学习顺序应用固定的合并规则
                    for pair, new_id in self.merges.items():
                        ids = merge(ids, pair, new_id)

                    result.extend(ids)

        return result

    def decode(self, ids):
        return decode(ids, self.vocab)

tokenizer = Tokenizer(loaded_vocab, loaded_merges, loaded_special_ids, full_pattern)
for sample in [train_text, "你好。\n", "", "ab  ab!"]:
    assert tokenizer.decode(tokenizer.encode(sample)) == sample
print("编解码检查通过")

编解码检查通过
